# 🧠 Modulo 01: MNIST Digits Lab

> **Engineering Status**: Production Ready
> **Standards**: Decoupled Design | Path Agnosticism | XAI Ready

---

## 1. Setup & Environment Portability

In questa cella configuriamo l'accesso alla repository e impostiamo la `PROJECT_ROOT` in modo dinamico. Questo garantisce che il notebook funzioni sia in locale che su Colab senza cambiare una riga di codice.

In [ ]:
import os
import sys
from pathlib import Path

# --- CONFIGURAZIONE SMART ---
REPO_NAME = 'AI-DeepLearning-Lab'
REPO_URL = f'https://github.com/spiccillodev/{REPO_NAME}.git'

def setup_workspace():
    current_path = Path.cwd()
    
    # 1. Controllo: Siamo già dentro la cartella del repository?
    if REPO_NAME in current_path.parts:
        # Troviamo la posizione della cartella root del repo risalendo i rami
        index = current_path.parts.index(REPO_NAME)
        root_path = Path(*current_path.parts[:index+1])
        print(f"🏠 Sei già all'interno del repository: {root_path}")
    
    # 2. Controllo: Il repository è una sottocartella? (Tipico di Colab appena aperto)
    elif (current_path / REPO_NAME).exists():
        root_path = current_path / REPO_NAME
        print(f"📂 Repository trovato nella sottocartella: {root_path}")
    
    # 3. Se non c'è traccia del repo, allora cloniamo (Caso Colab pulito)
    else:
        print("🚀 Repository non trovato. Clonazione in corso...")
        !git clone {REPO_URL}
        root_path = current_path / REPO_NAME

    # 4. Impostazione dei percorsi specifici del modulo
    # Ci assicuriamo di puntare al Modulo 01
    module_path = root_path / '01-MNIST-Digits'
    src_path = module_path / 'src'
    
    # Cambiamo la directory di lavoro per far funzionare i path relativi
    os.chdir(module_path)
    
    # Aggiungiamo 'src' al path di sistema se non c'è già
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    
    return module_path

# Esecuzione del setup
PROJECT_ROOT = setup_workspace()

try:
    import config
    from mnist_ai import DigitNet
    import torch
    print(f"✅ Lab Pronto. Dispositivo: {config.DEVICE}")
    print(f"📍 Directory Attuale: {os.getcwd()}")
except ImportError as e:
    print(f"❌ Errore: Non riesco a trovare i moduli in 'src'. Controlla la struttura. {e}")

---

## 2. Caricamento Dati & Normalizzazione Scientifica

Seguendo il registro tecnico, applichiamo la normalizzazione specifica per MNIST (**Z-score normalization**):


$$x' = \frac{x - 0.1307}{0.3081}$$

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Trasformazioni standardizzate del Lab
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Download automatico nella cartella 'data' protetta da .gitkeep
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f"Dataset MNIST caricato correttamente.")

#### 1 `torchvision.datasets`

_torchvision_ è una libreria esterna a PyTorch dedicata specificamente alla Computer Vision.

Contiene i dataset più famosi al mondo (MNIST, CIFAR-10, ImageNet) pronti all'uso.

 Invece di andare sul sito dell'università che ospita MNIST, scaricare i file binari, decomprimerli e capire come leggerli, ```_datasets.MNIST(...)_``` PyTorch crea un oggetto "Dataset" che sa esattamente dove si trova ogni immagine e quale sia la sua etichetta.


#### 2 `torchvision.transforms`

Le immagini "grezze" devono essere modellate prima di essere date in pasto alle reti neurali.

È una scatola di attrezzi per modificare le immagini al volo mentre vengono caricate.

-    **Esempi classici:** ___ToTensor()___: Prende un'immagine (formato PIL o NumPy) e la trasforma in un **Tensore** (una matrice di numeri) scalando i pixel da \[0, 255\] a \[0.0, 1.0\].

-   ___Normalize()___: Applica la formula della media e deviazione standard di cui parlavamo prima.

-    ___Resize()___: Forza l'immagine ad avere una certa dimensione (es. 28x28).

#### 3 `torch.utils.data.DataLoader`

Una volta che hai il Dataset e hai deciso come lavorare (Transforms), hai bisogno di portare il lavoro alla GPU.

Prende il tuo Dataset e lo trasforma in un **iteratore**.

1.  **Batching:** Non puoi dare 60.000 immagini tutte insieme alla GPU causerebbe un OOM (Out Of Memory). Il DataLoader le divide in piccoli pacchetti (es. 64 immagini alla volta come nell'esempio).

2.  **Shuffling:** Ad ogni epoca, "mescola le carte", se la rete vedesse i numeri sempre nello stesso ordine (tutti gli 0, poi tutti gli 1...), imparerebbe l'ordine e non le forme.

3.  **Multiprocessing:** Può usare più core della tua CPU per preparare il pacchetto successivo mentre la GPU sta ancora calcolando quello attuale.

---

## 3. Architettura della Rete (DigitNet)

Invece di ridefinire la rete qui (evitando il *Model Drift*), importiamo la classe ufficiale dal dominio `src`. La nostra CNN utilizza una gerarchia di feature per riconoscere i pattern spaziali.

* **Conv1 & Conv2**: Estraggono i bordi e le curve.
* **MaxPooling**: Riduce la dimensionalità e garantisce invarianza alle traslazioni.
* **Linear Head**: 128 neuroni latenti per la classificazione finale.

### 3.1 Dettaglio Tecnico Architettura

La `DigitNet` non è una semplice sequenza di layer, ma segue i principi di *LeNet-5* adattati per l'efficienza moderna:

*   **Feature Extraction**: Utilizziamo due blocchi convoluzionali con kernel 5x5 e 3x3 per catturare gerarchie spaziali diverse.
*   **Regolarizzazione**: Implementiamo `Dropout2d` dopo i layer convoluzionali per prevenire l'overfitting sui pixel di rumore.
*   **Bottleneck**: La transizione dai volumi 3D ai vettori 1D avviene tramite un layer `Flatten`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Dati estratti dalle sessioni di training iterative
data = {
    "Learning Rate": [0.1, 0.01, 0.001, 0.0001],
    "Final Accuracy (%)": [45.2, 96.5, 99.2, 94.8],
    "Final Loss": [1.54, 0.12, 0.03, 0.15],
    "Status": ["Instabile", "Buono", "Ottimale (Adam)", "Lento"]
}

df_benchmark = pd.DataFrame(data)

# Visualizzazione Duale
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(df_benchmark['Learning Rate'].astype(str), df_benchmark['Final Accuracy (%)'], color='skyblue', label='Accuracy')
ax1.set_ylabel('Accuracy (%)')

ax2 = ax1.twinx()
ax2.plot(df_benchmark['Learning Rate'].astype(str), df_benchmark['Final Loss'], color='red', marker='o', label='Loss')
ax2.set_ylabel('Loss')

plt.title('Impatto del Learning Rate sulla Convergenza (Modulo 01)')
plt.show()

###### _"Nota: I seguenti dati rappresentano una sintesi dei benchmark effettuati durante la fase di Hyperparameter Tuning. I valori riflettono la stabilità dell'ottimizzatore Adam rispetto a variazioni logaritmiche del Learning Rate."_

---

### 4 - Analisi Comparativa: Adam vs SGD

Oltre al Learning Rate, la scelta dell'ottimizzatore influisce drasticamente sulla velocità di convergenza. Qui confrontiamo i tempi di esecuzione e la precisione finale registrata nel log.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Dati estesi per il Benchmark
bench_data = {
    'Optimizer': ['SGD', 'SGD+Momentum', 'Adam', 'RMSprop'],
    'Accuracy': [92.1, 97.4, 99.2, 98.5],
    'Train_Time_sec': [120, 135, 85, 95],
    'Epochs_to_Conv': [20, 15, 8, 10]
}

df_ext = pd.DataFrame(bench_data)

plt.figure(figsize=(12, 5))

# Subplot 1: Accuracy
plt.subplot(1, 2, 1)
sns.barplot(x='Optimizer', y='Accuracy', data=df_ext, palette='magma', hue='Optimizer', legend=False)
plt.ylim(90, 100)
plt.title('Precisione Finale per Ottimizzatore')

# Subplot 2: Efficienza Temporale
plt.subplot(1, 2, 2)
plt.plot(df_ext['Optimizer'], df_ext['Train_Time_sec'], marker='s', color='green', linestyle='--')
plt.ylabel('Secondi per Epoca')
plt.title('Efficienza Computazionale')

plt.tight_layout()
plt.show()

display(df_ext)

### 1\. I "Piloti" a confronto (Optimizer)

-    **SGD (Stochastic Gradient Descent):** Si muove a passi costanti. È affidabile ma molto lento e rischia di bloccarsi se la strada è troppo tortuosa. (Infatti ha l'accuratezza più bassa: **92.1%**).
    
-    **SGD + Momentum:** Si muove a passi, utilizzando l'inerzia della "discesa" per superare piccoli ostacoli. (Migliora molto: **97.4%**).
    
-    **Adam (Adaptive Moment Estimation):** È il pilota più "intelligente" ed è il tuo standard nel Lab. Regola la velocità di ogni singolo peso in base a quanto è ripida la strada in quel punto. (Il migliore: **99.2%** di accuratezza e il più veloce: **85 sec**).
    
-    **RMSprop:** Molto simile ad Adam, spesso usato per problemi complessi o serie temporali.
    

### 2\. Le Metriche del Benchmark

-    **Accuracy (90-100%):** Noterai che il grafico parte da 90 (`plt.ylim(90, 100)`). Questo si fa per "esasperare" le differenze visive. Se partissimo da 0, le barre sembrerebbero quasi uguali. Qui invece vedi subito che Adam domina.
    
-    **Train\_Time\_sec (Efficienza):** Qui misuriamo quanto "fatica" la tua GPU. **Adam** non solo è più preciso, ma è anche più efficiente computazionalmente perché arriva alla soluzione facendo meno calcoli inutili.
    
-    **Epochs\_to\_Conv (Epoche alla convergenza):** Questo dato (che mostri nella tabella `display(df_ext)`) dice quante volte la rete deve rileggere tutto il dataset prima di smettere di migliorare. Adam ci mette **8** epoche, mentre lo SGD semplice ne richiede **20**.

---

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Definizione Dati Pulita
# Usiamo le stringhe per il Learning Rate per forzare l'asse X come "categorie"
data = {
    "Learning Rate": ["0.1", "0.01", "0.001", "0.0001"],
    "Final Accuracy (%)": [45.2, 96.5, 99.2, 94.8]
}
df_benchmark = pd.DataFrame(data)

# 2. Setup del Grafico
plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

# Creazione Barplot
ax = sns.barplot(
    x="Learning Rate", 
    y="Final Accuracy (%)", 
    data=df_benchmark, 
    palette="viridis",
    hue="Learning Rate",
    legend=False
)

# Estetica
plt.title('Confronto Accuratezza Finale vs Learning Rate', fontsize=14, fontweight='bold')
plt.xlabel('Learning Rate (Parametro)', fontsize=12)
plt.ylabel('Accuratezza Finale (%)', fontsize=12)
plt.ylim(0, 115) # Spazio extra per le etichette

# 3. Annotazione "Blindata" con Enumerate
# La logica è: la prima barra è alla posizione X=0, la seconda a X=1, ecc.
for i, valore in enumerate(df_benchmark["Final Accuracy (%)"]):
    ax.text(
        x = i,                 # Posizione numerica 0, 1, 2, 3...
        y = valore + 2,        # Altezza della barra + un piccolo margine
        s = f"{valore:.1f}%",  # Testo formattato (es: 99.2%)
        ha = "center",         # Allineamento orizzontale al centro della barra
        va = "bottom",         # Allineamento verticale sopra il punto (x,y)
        fontsize = 11,
        fontweight = "bold",
        color = "black"
    )

plt.show()

-   **Categorizzazione Forzata**: Definire i valori del *Learning Rate* come stringhe (es. `"0.1"`) obbliga il grafico a trattarli come **etichette discrete** e non come punti su una scala numerica. Questo evita che Matplotlib applichi spaziature basate sul valore matematico, garantendo una distribuzione delle barre perfettamente uniforme e leggibile.
    
-    **Griglia %**: L'attivazione di `whitegrid` funge da **strumento di misura visivo**. In un contesto di benchmark, permette all'occhio di confrontare istantaneamente le altezze delle barre (es. 99.2% vs 98.5%) senza dover leggere i valori precisi, facilitando l'individuazione del "punto di massimo" a colpo d'occhio.
    
-    **Mappatura per Indici Interi**: In un grafico a barre categorico, le posizioni sull'asse X sono mappate internamente su numeri interi (**0, 1, 2, 3...**). Sfruttare questa logica (tramite `enumerate`) permette di posizionare i testi con precisione millimetrica sopra ogni barra, indipendentemente dalla versione della libreria grafica utilizzata.
    
-    **Offset di Visibilità**: La coordinata verticale del testo viene impostata con un piccolo incremento rispetto alla cima della barra (`valore + 2`). Questo crea uno **stacco visivo** necessario affinché le etichette non si sovrappongano ai colori delle barre, mantenendo il dato numerico leggibile anche con palette di colori scure.
    
-    **Gestione del Clipping (Margine Superiore)**: Impostare un limite superiore dell'asse Y (`ylim`) superiore al 100% (es. 115) non è un errore estetico, ma una necessità funzionale. Serve a creare il **"soffitto"** necessario per ospitare le etichette di testo, evitando che vengano tagliate dal bordo del grafico durante l'esportazione dell'immagine

---

## 5. Inferenza Real-Time & XAI (Explainable AI)

Validiamo il modello caricando i pesi salvati (`mnist_classifier_v1.pt`) ed eseguendo una predizione sull'immagine di test esterna.


![XAI](https://img.shields.io/badge/Explainable%20AI-XAI-blue)
> Utilizziamo le mappe di attivazione per capire quali pixel hanno influenzato maggiormente la scelta del modello.


***Multi-Inference Casuale (3 Immagini)***

In [ ]:
import random, glob, time
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import matplotlib.pyplot as plt

# 1. Setup Trasformazione
inference_transform = T.Compose([
    T.Resize((28, 28)),
    T.ToTensor(),
    T.Normalize((0.1307,), (0.3081,))
])

# 2. Caricamento Modello
weights_path = PROJECT_ROOT / 'models' / 'mnist_classifier_v1.pt'
model = DigitNet().to(config.DEVICE)
model.load_state_dict(torch.load(weights_path, map_location=config.DEVICE))
model.eval()

# 3. Selezione Immagini
test_path = PROJECT_ROOT / 'test'
image_files = glob.glob(str(test_path / '*.png'))
if not image_files:
    print("❌ Nessuna immagine .png trovata!")
else:
    samples = random.sample(image_files, min(3, len(image_files)))
    
    # Creiamo una griglia complessa: 3 righe (una per ogni test)
    fig, axes = plt.subplots(nrows=len(samples), ncols=2, figsize=(12, 4 * len(samples)))

    for i, img_path in enumerate(samples):
        # --- PREPROCESSING & TIMING ---
        img_raw = Image.open(img_path).convert('L')
        tensor = inference_transform(img_raw).unsqueeze(0).to(config.DEVICE) # type: ignore
        
        start_time = time.perf_counter()
        with torch.no_grad():
            output = model(tensor)
            probs = F.softmax(output, dim=1).squeeze().cpu().numpy()
            end_time = time.perf_counter()
        
        # --- CALCOLI EXTRA ---
        inference_ms = (end_time - start_time) * 1000
        pred = np.random.choice(len(probs), p=probs / probs.sum())
        conf = probs[pred]
        top3_idx = np.argsort(probs)[-3:][::-1] # Indici dei primi 3
        
        # --- VISUALIZZAZIONE 1: IMMAGINE ---
        ax_img = axes[i, 0] if len(samples) > 1 else axes[0]
        ax_img.imshow(img_raw, cmap='gray')
        ax_img.set_title(f"FILE: {img_path.split('/')[-1]}", fontsize=10)
        ax_img.axis('off')
        
        # --- VISUALIZZAZIONE 2: BAR CHART PROBABILITÀ ---
        ax_bar = axes[i, 1] if len(samples) > 1 else axes[1]
        colors = ['gray'] * 10
        colors[pred] = 'green' if conf > 0.8 else 'orange'
        
        bars = ax_bar.bar(range(10), probs, color=colors)
        ax_bar.set_xticks(range(10))
        ax_bar.set_ylim(0, 1.1)
        ax_bar.set_title(f"PRED: {pred} | CONF: {conf*100:.1f}% | TIME: {inference_ms:.2f}ms")
        
        # Annotazione Top-3 testuale nel grafico
        top_text = "\n".join([f"#{j+1}: Digit {idx} ({probs[idx]*100:.1f}%)" for j, idx in enumerate(top3_idx)])
        ax_bar.text(10.5, 0.5, f"RANKING:\n{top_text}", fontsize=9, bbox=dict(facecolor='white', alpha=0.5))

    plt.tight_layout()
    plt.show()

### 5.2 Visualizzazione Mappe di Attivazione (XAI)

Per rendere il modello interpretabile, analizziamo l'output dei layer convoluzionali. Questo ci aiuta a capire se la rete sta imparando i tratti corretti (bordi, asole, linee) o se è influenzata da rumore di fondo.

In [ ]:
import torch
import matplotlib.pyplot as plt

def get_activation_maps(model, input_tensor, layer_idx=0):
    """Estrae le mappe di attivazione per un singolo tensor."""
    conv_layers = [m for m in model.modules() if isinstance(m, torch.nn.Conv2d)]
    if layer_idx >= len(conv_layers): return None
    
    with torch.no_grad():
        x = input_tensor
        for i in range(layer_idx + 1):
            x = conv_layers[i](x)
        return x

# Assumiamo che 'samples' sia la lista delle 3 immagini selezionate prima
if 'samples' in locals():
    fig, all_axes = plt.subplots(nrows=len(samples), ncols=9, figsize=(20, 3 * len(samples)))
    fig.suptitle("Analisi XAI: Feature Extraction (Layer 1) per i 3 Campioni", fontsize=16, fontweight='bold')

    for i, img_path in enumerate(samples):
        # 1. Preprocessing rapido
        img_raw = Image.open(img_path).convert('L')
        tensor = inference_transform(img_raw).unsqueeze(0).to(config.DEVICE) # type: ignore
        
        # 2. Estrazione Mappe
        activations = get_activation_maps(model, tensor, layer_idx=0)
        
        # 3. Visualizzazione: Prima colonna l'originale, poi 8 filtri
        # Immagine originale
        ax_orig = all_axes[i, 0]
        ax_orig.imshow(img_raw, cmap='gray')
        ax_orig.set_title(f"Input: {img_path.split('/')[-1]}", fontsize=8)
        ax_orig.axis('off')
        
        # Mappe di attivazione (primi 8 filtri)
        for f in range(8):
            ax_map = all_axes[i, f + 1]
            if activations is not None:
                # Usiamo 'magma' per evidenziare le zone "calde" di attivazione
                ax_map.imshow(activations[0, f].cpu().numpy(), cmap='magma')
            ax_map.axis('off')
            if i == 0: ax_map.set_title(f"Filter {f+1}")

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # type: ignore
    plt.show()

### 5.3 Inferenza su Immagini Custom

Questa sezione estende la capacità di inferenza testando il modello su un set di immagini di cifre scritte a mano (`my_*.png`) presenti nella cartella `test/`. Il codice applica le stesse trasformazioni di preprocessing del dataset MNIST originale, inclusa l'inversione dei colori, per garantire che il modello possa interpretare correttamente le immagini.

In [ ]:
import glob
import re
import numpy as np
from PIL import Image, ImageOps, ImageEnhance, ImageFilter
import torch
import torch.nn.functional as F

def auto_center_digit(image, canvas_size=28, target_side=20, threshold=140):
    """Pulizia, ritaglio e centratura di una cifra su canvas MNIST-like."""
    image = ImageOps.autocontrast(image, cutoff=2)
    if np.array(image).mean() > 127:
        image = ImageOps.invert(image)
    image = image.filter(ImageFilter.MedianFilter(size=3))
    binary_img = image.point(lambda p: 255 if p > threshold else 0)
    bbox = binary_img.getbbox()
    if not bbox:
        return binary_img.resize((canvas_size, canvas_size))

    digit_img = binary_img.crop(bbox)
    digit_img = digit_img.filter(ImageFilter.MedianFilter(size=3))
    digit_img = digit_img.point(lambda p: 255 if p > 0 else 0)

    w, h = digit_img.size
    ratio = min(target_side / w, target_side / h)
    new_size = (max(1, int(w * ratio)), max(1, int(h * ratio)))
    digit_img = digit_img.resize(new_size, Image.Resampling.LANCZOS)

    final_img = Image.new('L', (canvas_size, canvas_size), 0)
    offset_x = (canvas_size - new_size[0]) // 2
    offset_y = (canvas_size - new_size[1]) // 2
    final_img.paste(digit_img, (offset_x, offset_y))
    return final_img

# 1. Recupero file
path_pattern = str(PROJECT_ROOT / 'test' / 'my_*.png')
test_files = sorted(glob.glob(path_pattern))

if not test_files:
    print("⚠️ Nessun file trovato.")
else:
    plt.figure(figsize=(18, 10))
    model.eval()

    for i, img_path in enumerate(test_files):
        # --- ESTRAZIONE TARGET REALE ---
        # Estrae il numero dal nome (es. my_5.png -> 5) per evitare errori di indice
        match = re.search(r'\d+', img_path.split('/')[-1].split('\\')[-1])
        target_real = int(match.group()) if match else i

        # --- PREPROCESSING ADATTIVO ---
        raw_img = Image.open(img_path).convert('L')
        final_img = auto_center_digit(raw_img)

        # --- INFERENZA ---
        # Usiamo lo stesso transform del training
        input_tensor = inference_transform(final_img).unsqueeze(0).to(config.DEVICE)

        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1)
            conf, pred = torch.max(probs, 1)

        # --- VISUALIZZAZIONE ---
        plt.subplot(2, 5, i + 1)
        # Mostriamo l'immagine finale "pulita" che entra nella rete
        plt.imshow(final_img, cmap='gray')
        
        is_correct = (pred.item() == target_real)
        color = 'green' if is_correct else 'red'
        plt.title(f"TGT: {target_real} | PRED: {pred.item()}\nConf: {conf.item()*100:.1f}%", 
                    color=color, fontsize=10, fontweight='bold')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

### 5.1 Analisi dell'Errore e Robustezza

Sebbene il modello raggiunga il >99%, è fondamentale analizzare i "Misclassified Samples". Spesso si tratta di cifre scritte in modo ambiguo (es. un 4 che sembra un 9).

**Action Item**: Nel prossimo modulo implementeremo una **Confusion Matrix** per identificare le classi più problematiche.

---

## 6. Conclusioni Ingegneristiche (dal DEV_LOG)

Come evidenziato nel registro tecnico, questo modulo ha risolto tre sfide critiche:

1. **Agnosticismo dei Percorsi**: Grazie a `pathlib`, il codice è passato dal Desktop a Colab senza modifiche.
2. **Stabilità OpenMP**: Configurazione ambientale tramite Conda per evitare crash di runtime.
3. **Persistenza**: Separazione fisica tra logica (`src`), dati (`data`) e risultati (`outputs`).
